# Análisis Exploratorio de Datos y Machine Learning Básico
Notebook genérico para cargar datos, explorarlos y entrenar modelos básicos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## Carga de datos

In [ ]:
ruta = 'datos.csv'  # Cambiar por su archivo
df = pd.read_csv(ruta)
df.head()

## Exploración inicial

In [ ]:
print(df.shape)
df.info()
df.describe(include='all').T

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
if len(num_cols) > 0:
    df[num_cols].hist(figsize=(12,8))
    plt.tight_layout()
    plt.show()

In [ ]:
if len(num_cols) > 1:
    plt.figure(figsize=(8,6))
    sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
    plt.show()

## Machine Learning básico

In [ ]:
target = 'objetivo'  # Cambiar por la variable objetivo
X = df.drop(columns=[target])
y = df[target]

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

num_features = X.select_dtypes(include=np.number).columns
cat_features = X.select_dtypes(exclude=np.number).columns

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred))